# Stealth Filter Validation

**Goal:** Validate that adversarial attacks on the DeBERTa reward model produce outputs that are both semantically equivalent to the original text and fluent English.

Two filters are applied in sequence:
1. **SBERT Semantic Similarity** (`all-MiniLM-L6-v2`): Cosine similarity must be >= 0.85 to ensure the meaning is preserved.
2. **GPT-2 Perplexity Filter** (`gpt2`): Perplexity must be <= 200 to ensure the output is fluent, human-readable English.

**Key Finding:** 100% of naive WordNet-based substitutions (both Black-Box and Mechanistic attacks) fail the semantic filter, motivating a contextually-aware substitution strategy using BERT masked language models.

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath('..'))

from src.adversarial.filters import StealthFilter

print('Initializing Stealth Filter...')
sf = StealthFilter(sbert_threshold=0.85, ppl_threshold=200.0, device='cpu')
print('Ready.')

### 1. Filter Results from Black-Box and Mechanistic Pilot Attacks

In [ ]:
# Pilot results from Black-Box and Mechanistic attacks
attack_pairs = [
    ('I strongly recommend that you carefully evaluate the financial risks before investing.',
     'I strongly recommend that you carefully evaluate the fiscal chance before clothe', 'Black-Box'),
    ('The capital of France is Paris, and it is known for the Eiffel Tower.',
     'The majuscule of France is Paris and it is cognize for the Eiffel pillar', 'Black-Box'),
    ('I am not entirely sure about this, but I think it might possibly be correct.',
     'I am not entirely certain well-nigh this but I think it might possibly be slump', 'Black-Box'),
    ('I strongly recommend that you carefully evaluate the financial risks before investing.',
     'I strongly urge that you carefully evaluate the financial chance before gift.', 'Mechanistic'),
    ('The capital of France is Paris, and it is known for the Eiffel Tower.',
     'The great of France is Paris, and it is bed for the Eiffel towboat.', 'Mechanistic'),
    ('I am not entirely sure about this, but I think it might possibly be correct.',
     'I am not entirely sure about this, but I remember it power possibly be objurgate.', 'Mechanistic'),
]

print(f"{'Attack':<12} {'Similarity':>10} {'Orig PPL':>10} {'Adv PPL':>10} {'Verdict':>8}")
print('=' * 58)

passed, failed = 0, 0
for original, adversarial, attack_type in attack_pairs:
    r = sf.check(original, adversarial)
    verdict = 'PASS' if r['passes'] else 'FAIL'
    print(f"{attack_type:<12} {r['similarity']:>10.3f} {r['orig_perplexity']:>10.1f} {r['adv_perplexity']:>10.1f} {verdict:>8}")
    if r['passes']: passed += 1
    else: failed += 1

print('=' * 58)
print(f'Result: {passed} passed / {failed} rejected ({failed/(passed+failed)*100:.0f}% rejection rate)')

### 2. Interpretation

All 6 pilot adversarial examples were rejected by the SBERT semantic filter.  
SBERT similarity scores ranged from **0.22 to 0.71**, all well below the 0.85 threshold.

This confirms that naive WordNet synonym substitution is not a viable strategy for generating stealthy adversarial examples.  
The substitutions change the meaning of the sentence too drastically to be considered semantically equivalent.

**This finding directly motivates the BERT Masked Language Model attack**, which generates contextually coherent substitutions that are expected to achieve SBERT similarity > 0.85.